
**Dataset:** *Global Power Plant Database v1.3.0* by the **World Resources Institute** (`global_power_plant_database.csv`).  
≈ 34 936 power plants worldwide with capacity, fuel type, location, commissioning year and yearly generation.

**Pipeline**
1. Import & cleaning  
2. Exploratory data analysis (EDA)  
3. Statistical analysis (NumPy + SciPy)  
4. Time-series analysis  
5. Advanced visualization  
6. Matrix operations in a real context  
7. NumPy ↔ Pandas ↔ Matplotlib integration  

All comments are in English.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', 60)
sns.set_theme(style='whitegrid')

BASE_DIR = os.getcwd()
print('Working directory:', BASE_DIR)

## 1. Data Import and Cleaning

In [ ]:
df = pd.read_csv(os.path.join(BASE_DIR, 'global_power_plant_database.csv'),
                 low_memory=False)
print('Shape:', df.shape)
df.head()

In [ ]:
# Missing values per column (top 15)
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'missing': missing, 'pct': missing_pct})
missing_report.head(15)

In [ ]:
# Cast numerical columns with pandas + NumPy (coerce -> NaN on bad values)
num_cols = ['capacity_mw', 'latitude', 'longitude', 'commissioning_year',
            'year_of_capacity_data']
gen_cols = [c for c in df.columns if c.startswith('generation_gwh_')]
est_cols = [c for c in df.columns if c.startswith('estimated_generation_gwh_')]

for c in num_cols + gen_cols + est_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# `commissioning_year` is supposed to be an integer year -> clip out-of-range values
df.loc[df['commissioning_year'] < 1880, 'commissioning_year'] = np.nan
df.loc[df['commissioning_year'] > 2030, 'commissioning_year'] = np.nan

# Drop rows with no capacity (we need it everywhere)
df = df.dropna(subset=['capacity_mw', 'primary_fuel'])

print('Shape after cleaning:', df.shape)
print('Number of unique countries :', df['country_long'].nunique())
print('Number of unique fuels     :', df['primary_fuel'].nunique())

## 2. Exploratory Data Analysis

In [ ]:
# Descriptive stats on the main numerical columns
df[['capacity_mw', 'commissioning_year', 'latitude', 'longitude']].describe().round(2)

In [ ]:
# Number of plants and total installed capacity per primary fuel
by_fuel = df.groupby('primary_fuel').agg(
    n_plants      = ('gppd_idnr',  'count'),
    total_mw      = ('capacity_mw','sum'),
    mean_mw       = ('capacity_mw','mean'),
    median_mw     = ('capacity_mw','median'),
    std_mw        = ('capacity_mw','std'),
).round(2).sort_values('total_mw', ascending=False)
by_fuel

In [ ]:
# Top 15 countries by installed capacity
by_country = df.groupby('country_long').agg(
    n_plants  = ('gppd_idnr',  'count'),
    total_mw  = ('capacity_mw','sum'),
).sort_values('total_mw', ascending=False).head(15)
by_country

## 3. Statistical Analysis

In [ ]:
# Use NumPy directly on the capacity array per fuel
fuels = by_fuel.index.tolist()
rows = []
for fuel in fuels:
    arr = df.loc[df['primary_fuel'] == fuel, 'capacity_mw'].to_numpy()
    rows.append({
        'fuel'    : fuel,
        'count'   : arr.size,
        'mean'    : np.mean(arr),
        'median'  : np.median(arr),
        'std'     : np.std(arr, ddof=1),
        'q95'     : np.quantile(arr, 0.95),
        'log_mean': np.log1p(arr).mean(),   # log-mean is more robust to the heavy tail
    })
stats_fuel = pd.DataFrame(rows).round(2)
stats_fuel.head(10)

In [ ]:
# Hypothesis test: do mean capacities differ across fuel types?
# H0: every fuel has the same mean capacity.
groups = [df.loc[df['primary_fuel'] == f, 'capacity_mw'].dropna().to_numpy() for f in fuels]
# Keep fuels with at least 30 plants for a sensible ANOVA
groups = [g for g in groups if g.size >= 30]

f_stat, p_value = stats.f_oneway(*groups)
print(f'ANOVA on capacity by fuel  ->  F = {f_stat:.2f}, p = {p_value:.3e}')
if p_value < 0.05:
    print('  -> Reject H0: at least one fuel type has a significantly different mean capacity.')
else:
    print('  -> Fail to reject H0.')

In [ ]:
# Focus comparison: Coal vs Solar (very different technologies, expected different means)
coal  = df.loc[df['primary_fuel'] == 'Coal',  'capacity_mw'].to_numpy()
solar = df.loc[df['primary_fuel'] == 'Solar', 'capacity_mw'].to_numpy()
t_stat, p_value = stats.ttest_ind(coal, solar, equal_var=False)
print(f'Welch t-test Coal vs Solar  ->  t = {t_stat:.2f}, p = {p_value:.3e}')
print(f'Mean capacity Coal  : {coal.mean():,.1f} MW')
print(f'Mean capacity Solar : {solar.mean():,.1f} MW')

## 4. Time-Series Analysis (commissioning years & fuel mix evolution)

In [ ]:
# Number of newly commissioned plants and added capacity per year
by_year = (df.dropna(subset=['commissioning_year'])
             .assign(year=lambda d: d['commissioning_year'].astype(int))
             .groupby('year')
             .agg(n_plants=('gppd_idnr','count'),
                  added_mw=('capacity_mw','sum')))
by_year.tail(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
by_year['n_plants'].plot(ax=axes[0], color='steelblue')
axes[0].set_title('Plants commissioned per year')
axes[0].set_ylabel('Number of plants')

by_year['added_mw'].plot(ax=axes[1], color='seagreen')
axes[1].set_title('New capacity added per year (MW)')
axes[1].set_ylabel('MW added')
plt.tight_layout()
plt.show()

In [ ]:
# Evolution of the fuel mix: capacity added per (year, primary_fuel)
fuel_mix = (df.dropna(subset=['commissioning_year'])
              .assign(year=lambda d: d['commissioning_year'].astype(int))
              .groupby(['year', 'primary_fuel'])['capacity_mw'].sum()
              .unstack(fill_value=0))

# Keep only the main fuel types and the last 50 years for readability
main_fuels = ['Coal', 'Gas', 'Hydro', 'Nuclear', 'Oil', 'Solar', 'Wind', 'Biomass']
main_fuels = [f for f in main_fuels if f in fuel_mix.columns]
fuel_mix = fuel_mix.loc[1970:2020, main_fuels]

fig, ax = plt.subplots(figsize=(12, 5))
fuel_mix.plot.area(ax=ax, colormap='tab20', alpha=0.85)
ax.set_title('Annual capacity added per primary fuel (1970-2020)')
ax.set_ylabel('MW added')
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

## 5. Advanced Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
by_country['total_mw'].head(15).iloc[::-1].plot(
    kind='barh', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Top 15 countries by total installed capacity')
axes[0].set_xlabel('MW')

by_fuel['total_mw'].head(10).plot(
    kind='bar', ax=axes[1], color='seagreen', edgecolor='black')
axes[1].set_title('Top 10 fuels by total installed capacity')
axes[1].set_ylabel('MW')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Worldwide geographic scatter, color = fuel, size = log(capacity)
main_fuels_set = set(main_fuels)
geo = df[df['primary_fuel'].isin(main_fuels_set)].dropna(subset=['latitude', 'longitude'])

plt.figure(figsize=(14, 7))
palette = sns.color_palette('tab10', n_colors=len(main_fuels))
for color, fuel in zip(palette, main_fuels):
    sub = geo[geo['primary_fuel'] == fuel]
    plt.scatter(sub['longitude'], sub['latitude'],
                s=np.log1p(sub['capacity_mw']) * 2,
                alpha=0.4, label=fuel, color=color)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Global distribution of power plants (size ∝ log capacity)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Density of plants over the globe: hexbin gives a smoother density view than scatter
plt.figure(figsize=(13, 6))
plt.hexbin(df['longitude'], df['latitude'], gridsize=60, cmap='inferno', mincnt=1)
plt.colorbar(label='Number of plants')
plt.title('Power-plant density worldwide (hexbin)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()

In [ ]:
# Capacity distribution: linear vs log scale (heavy right tail)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['capacity_mw'], bins=80, color='steelblue', edgecolor='black')
axes[0].set_title('Capacity (MW) — linear x-axis')
axes[0].set_xlabel('Capacity (MW)')

axes[1].hist(np.log1p(df['capacity_mw']), bins=80, color='steelblue', edgecolor='black')
axes[1].set_title('log1p(Capacity) — log x-axis')
axes[1].set_xlabel('log1p(capacity_mw)')
plt.tight_layout()
plt.show()

## 6. Matrix Operations in a Real Context

We build a **fuel × country** matrix (rows = top fuels, columns = top countries, value = total capacity in MW), then run linear-algebra operations on it.

In [ ]:
top_fuels   = by_fuel.index[:8].tolist()
top_countries_idx = by_country.index[:8].tolist()

mat = (df[df['primary_fuel'].isin(top_fuels) & df['country_long'].isin(top_countries_idx)]
         .pivot_table(index='primary_fuel',
                      columns='country_long',
                      values='capacity_mw',
                      aggfunc='sum',
                      fill_value=0)
         .loc[top_fuels, top_countries_idx])
print('Capacity matrix (MW):')
mat.round(0).astype(int)

In [ ]:
# Eigen-decomposition of the matrix multiplied by its transpose (always square + symmetric)
M = mat.to_numpy().astype(float)
MtM = M @ M.T                           # (fuels x fuels)
eigvals, eigvecs = np.linalg.eigh(MtM)   # eigh because MtM is symmetric -> real eigenvalues

print('Eigenvalues (sorted desc):')
print(np.round(eigvals[::-1], 2))

# Variance share carried by each principal direction
share = eigvals[::-1] / eigvals.sum()
print('\nVariance share per component:')
print(np.round(share, 4))

### What do these eigenvalues / eigenvectors mean here?
- The matrix `M` measures how each fuel is distributed across countries.
- `M @ Mᵀ` measures **how similar two fuels are in their geographic distribution** (a kind of covariance between fuel rows).
- The **largest eigenvalue** tells us the dominant direction in fuel space — usually the *general size effect* (big countries have big installed capacity in every fuel).
- The **next eigenvalues** reveal *specific patterns*: e.g. *Coal vs Renewables*, *Hydro-rich countries vs Gas-rich countries*, etc.
- The associated **eigenvectors** are the directions along which countries differentiate themselves.
- This is exactly the principle behind **PCA**: project the original data onto the first few eigenvectors to keep the most informative structure.

## 7. Integrating NumPy with Pandas and Matplotlib

In [ ]:
# Example 1 — NumPy boolean mask to perform a complex filter inside Pandas
lat  = df['latitude'].to_numpy()
lon  = df['longitude'].to_numpy()
cap  = df['capacity_mw'].to_numpy()
fuel = df['primary_fuel'].to_numpy()

# Large renewable plants in the northern hemisphere
mask = (lat > 0) & (cap > 100) & np.isin(fuel, ['Solar', 'Wind', 'Hydro'])
selection = df[mask]
print('Large northern-hemisphere renewable plants (>100 MW):', selection.shape[0])
selection.head()

In [ ]:
# Example 2 — NumPy vectorized computation injected as a new Pandas column
# Approximate "distance from the equator" used as a renewable-potential proxy.
df['abs_lat']       = np.abs(df['latitude'])
df['lat_bucket']    = np.digitize(df['abs_lat'], bins=np.arange(0, 91, 10))
df['log_capacity']  = np.log1p(df['capacity_mw'])

df[['country_long', 'primary_fuel', 'capacity_mw',
    'abs_lat', 'lat_bucket', 'log_capacity']].head()

In [ ]:
# Example 3 — combining NumPy + Pandas + Matplotlib for an informative plot
# Mean log capacity per latitude bucket, faceted by fuel (top 4 fuels)
top4 = by_fuel.index[:4].tolist()
subset = df[df['primary_fuel'].isin(top4)]

agg = subset.groupby(['lat_bucket', 'primary_fuel'])['log_capacity'].mean().unstack()

fig, ax = plt.subplots(figsize=(10, 5))
agg.plot(marker='o', ax=ax)
ax.set_title('Mean log(capacity_mw) per 10° latitude band, top 4 fuels')
ax.set_xlabel('Latitude band (1 = 0-10°, 2 = 10-20°, ...)')
ax.set_ylabel('log1p(capacity_mw)')
ax.legend(title='primary_fuel')
plt.show()

## 8. Summary of Findings

- **Coverage** — the database contains ~35 000 plants in ~160 countries, spanning fossil (Coal, Gas, Oil) and renewable (Hydro, Solar, Wind, Biomass, Geothermal) sources.
- **Capacity is heavily right-skewed**: most plants are small (<100 MW) but a tail of very large plants (>1 GW), mostly coal, nuclear and hydro, dominates the total installed capacity.
- **Mean capacities differ very significantly between fuels** (ANOVA p ≈ 0). A simple Welch t-test confirms that coal plants are on average **much larger** than solar plants — a structural difference between centralized fossil generation and distributed renewables.
- **Time trend** — capacity additions accelerate after the 1980s. The fuel-mix area chart shows the **rise of solar and wind from the 2000s onward**, and a clear plateau/decline of coal additions in the most recent years.
- **Geography** — plant density is highest in Europe, eastern USA, India and eastern China; the hexbin density map makes that obvious.
- **Matrix analysis** — the fuel × country capacity matrix has a dominant first eigenvalue (a *size effect*: large countries have a lot of every fuel) and meaningful secondary components separating fossil-heavy countries from renewable-leaning ones.
- **NumPy + Pandas + Matplotlib integration** — NumPy boolean masks let us express complex filters (`(lat > 0) & (cap > 100) & np.isin(fuel, [...])`) and vectorized math (`np.log1p`, `np.digitize`, `np.abs`) creates useful derived columns that we can directly plot.

### Limits
- A non-negligible fraction of plants have a missing `commissioning_year` (selection bias possible for the time-series part).
- Yearly generation columns are mostly missing outside the OECD: the analysis focuses on **installed capacity**, which is more complete.
- Lat/long coordinates are approximate for some entries (geolocation source varies).

---
**End of Daily Challenge — Day 2.** Don't forget to push to GitHub.